In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.1f}с] {m}", flush=True)
os.makedirs("/kaggle/working/src", exist_ok=True); os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
print("ядер:", os.cpu_count(), flush=True)

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")[["id1", "id2"]]
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
# Тест сопоставим с matches + items_human, то есть примерно 365 тысяч пар: множим замеры.
SCALE = 365000 / len(pairs)
log(f"пар {len(pairs):,}, карточек {len(items):,}, множитель до теста {SCALE:.2f}")
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
known = sorted(set(items.category.astype(str)))
timings = {}
def stage(name, fn):
    gc.collect(); t = time.perf_counter(); r = fn()
    d = time.perf_counter() - t; timings[name] = d
    log(f"  {name:<34} {d:7.1f}с   -> на тесте ~{d*SCALE:7.1f}с")
    return r

from src.cross_encoder import build_product_texts
from src.pair_features import build_matrix_parallel
from src.measure_features import measures, compare_measures, MEASURE_FEATURES
from src.features import extract_model_features
from src.model import BoostedPairModel

used = pd.unique(np.concatenate([pairs.id1.to_numpy(), pairs.id2.to_numpy()]))
texts = stage("тексты для энкодера", lambda: build_product_texts(items[items.id.isin(used)], "compact"))
X = stage("парные признаки (параллельно)",
          lambda: build_matrix_parallel(items, pairs, known_categories=known, with_neighbours=False))
def do_measures():
    parsed = {int(i): measures(a) for i, a in zip(items.id, items.attributes)}
    empty = measures(None)
    return np.array([[r[n] for n in MEASURE_FEATURES] for r in
                     (compare_measures(parsed.get(int(a), empty), parsed.get(int(b), empty))
                      for a, b in zip(pairs.id1, pairs.id2))], dtype=np.float32)
MX = stage("величины (один поток)", do_measures)
legacy = stage("структурные 168 признаков", lambda: extract_model_features(pairs, items))
_ = stage("два бустинга по ним", lambda: (
    BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat),
    BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)))
total = sum(timings.values())
log(f"\nИТОГО CPU: {total:.1f}с -> на тесте ~{total*SCALE:.1f}с (без энкодеров)")
log("доли: " + ", ".join(f"{k} {v/total:.0%}" for k, v in sorted(timings.items(), key=lambda kv: -kv[1])))
json.dump({k: v for k, v in timings.items()}, open("/kaggle/working/timings.json", "w"), ensure_ascii=False, indent=1)
